# Video 360° e Proiezioni Equirettangolari

## Obiettivi della lezione

In questo laboratorio impareremo a:

1. Caricare ed ispezionare un video a 360° codificato in proiezione **equirettangolare (ERP)**;
2. Estrarre frame rappresentativi ed osservare empiricamente le distorsioni tipiche del formato;
3. Convertire un frame ERP in proiezione **cubemap** con `py360convert`;
4. Estrarre **viewport** prospettiche arbitrarie (quello che vedrebbe un utente con un visore VR);

**Prerequisiti:** Lezione precedente (lettura video con `cv2.VideoCapture`, scrittura con `cv2.VideoWriter`). Non ripetiamo qui la sintassi di base già vista: la richiamiamo solo dove necessario.

**Strumenti:** Python, OpenCV, `py360convert`.

In [ ]:
# Import delle librerie necessarie per l'intera lezione
import os
import time
import math

import numpy as np
import cv2
import matplotlib.pyplot as plt

import py360convert  # pip install py360convert

print("OpenCV:", cv2.__version__)
print("py360convert:", py360convert.__version__)

## 1. Caricamento e ispezione di un video equirettangolare

### Richiamo teorico: la proiezione equirettangolare (ERP)

Un video 360° cattura l'intera sfera visiva attorno alla camera. Per poterlo memorizzare e comprimere con i codec video standard (pensati per immagini rettangolari planari), la sfera viene "srotolata" su un piano tramite una **proiezione**. La più diffusa è la **proiezione equirettangolare (Equirectangular Projection, ERP)**:

- la **longitudine** (angolo orizzontale attorno alla sfera, da −180° a +180°) è mappata linearmente sulla coordinata **x** del frame;
- la **latitudine** (angolo verticale, da −90° al polo sud a +90° al polo nord) è mappata linearmente sulla coordinata **y**;
- ne consegue che l'aspect ratio standard di un frame ERP è **2:1** (360° di longitudine contro 180° di latitudine).

Questa mappatura è semplice da implementare ed è comoda perché produce un frame rettangolare "normale" per codec e contenitori esistenti. Il prezzo da pagare, come vedremo empiricamente, è una **distorsione di campionamento fortemente non uniforme**: un'area della sfera vicina al polo occupa, in ERP, la stessa larghezza orizzontale (in pixel) di un'area equivalente all'equatore, pur rappresentando una porzione di superficie sferica molto più piccola. Al polo stesso — un singolo punto geometrico — corrisponde un'intera riga di pixel.

![EPR](imgs/epr.png)
![EPR](imgs/epr2.png)

### Dalle coordinate sferiche ai pixel

Un punto della sfera visiva, cioè una direzione di vista, è individuato da due angoli:

- la **longitudine** $\lambda$, angolo orizzontale misurato dalla direzione frontale, in $[-180°, +180°]$ (positiva verso destra);
- la **latitudine** $\varphi$, angolo verticale misurato dall'orizzonte, in $[-90°, +90°]$ (positiva verso l'alto).

Un frame ERP di $W \times H$ pixel assegna a questi angoli le coordinate $(x, y)$, con l'origine nell'angolo **in alto a sinistra** e l'asse $y$ rivolto **verso il basso**, come in tutte le immagini digitali. La mappatura è lineare su entrambi gli assi, quindi ogni coordinata si ricava con una proporzione.

#### Asse orizzontale

L'intervallo di longitudine (ampio $360°$) va distribuito sulla larghezza $W$. Prima lo si trasla in modo che parta da zero, poi lo si scala:

$$x = \frac{\lambda + 180°}{360°} \cdot W$$

- $\lambda = -180°$ dà $x = 0$ (bordo sinistro);
- $\lambda = 0°$ dà $x = W/2$ (centro del frame, la direzione frontale);
- $\lambda = +180°$ dà $x = W$, che coincide con il bordo sinistro: $+180°$ e $-180°$ sono **lo stesso meridiano**, alle spalle dell'osservatore. I bordi sinistro e destro del frame sono quindi contigui sulla sfera, e la coordinata $x$ va intesa **modulo $W$**.

#### Asse verticale

L'intervallo di latitudine (ampio $180°$) va distribuito sull'altezza $H$. Poiché $y$ cresce verso il basso mentre la latitudine cresce verso l'alto, la si misura **a partire dal polo nord**, cioè si usa $90° - \varphi$:

$$y = \frac{90° - \varphi}{180°} \cdot H$$

- $\varphi = +90°$ (polo nord) dà $y = 0$, la prima riga;
- $\varphi = 0°$ (orizzonte) dà $y = H/2$, la riga centrale;
- $\varphi = -90°$ (polo sud) dà $y = H$, il bordo inferiore.

A differenza della longitudine, la latitudine **non è periodica**: oltre il polo nord non si ricomincia dal polo sud, ma si scende dall'altra parte del polo. Per questo $y$ non si tratta modulo $H$: i valori fuori da $[0, H]$ indicano semplicemente un punto non valido.

#### Risoluzione angolare

Dalle due formule si ricava quanti pixel corrispondono a un grado:

$$\frac{W}{360°} \text{ in orizzontale}, \qquad \frac{H}{180°} \text{ in verticale}$$

Nel formato 2:1 ($W = 2H$) i due valori coincidono: un grado vale lo stesso numero di pixel in entrambe le direzioni. È la ragione geometrica dell'aspect ratio 2:1, discussa più avanti. Il valore orizzontale, però, **non dipende dalla latitudine**: è l'origine della distorsione che analizzeremo nella Sezione 2.

#### La trasformazione inversa

Le formule si invertono facilmente, per sapere quale direzione rappresenta un dato pixel:

$$\lambda = \frac{x}{W} \cdot 360° - 180°, \qquad \varphi = 90° - \frac{y}{H} \cdot 180°$$

#### Dai valori continui agli indici di pixel

Le formule producono valori reali, mentre i pixel hanno indici interi da $0$ a $W-1$ e da $0$ a $H-1$. Ci sono due convenzioni, da non confondere:

- per sapere **in quale pixel cade** una direzione si prende la parte intera, $\lfloor x \rfloor$ e $\lfloor y \rfloor$. Per $\varphi = -90°$ si ottiene $y = H$, che va riportato all'ultima riga $H-1$;
- per sapere **quale direzione rappresenta il centro** del pixel di indici $(i, j)$ si usano $x = i + 0{,}5$ e $y = j + 0{,}5$ nelle formule inverse.


#### Esempio numerico

Per il video di questo laboratorio ($W = 4320$, $H = 2160$) un grado vale $4320 / 360 = 2160 / 180 = 12$ pixel. La direzione $\lambda = 90°$, $\varphi = 45°$ (a destra e a metà altezza verso lo zenit) corrisponde a:

$$x = \frac{90 + 180}{360} \cdot 4320 = 3240, \qquad y = \frac{90 - 45}{180} \cdot 2160 = 540$$

cioè a tre quarti della larghezza e a un quarto dell'altezza del frame. Nella Sezione 2 tradurremo queste formule in codice con la funzione `lonlat_to_xy`.

Il video a 360 gradi che verrà usato in questo laboratorio è possibile scaricarlo da qui

https://drive.google.com/file/d/1CQAkqGUBiwwPxGlMcsVFpgbUdetMMjIf/view?usp=sharing

È possibile scaricare altri video da qui 

https://www.pexels.com/it-IT/cerca/videos/360/?orientation=landscape

In [ ]:
ERP_VIDEO_PATH = "sample.mp4"
ERP_IMAGE_PATH = "frame.jpg"

In [ ]:
# --- Caricamento e ispezione del video (riuso di quanto visto in Lezione 4) --
cap = cv2.VideoCapture(ERP_VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Impossibile aprire il video: {ERP_VIDEO_PATH}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_video = cap.get(cv2.CAP_PROP_FPS)
n_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Risoluzione: {width} x {height}")
print(f"Aspect ratio: {width / height:.3f}")   # verificare empiricamente se è 2.0
print(f"FPS: {fps_video}")
print(f"Numero di frame: {n_total_frames}")

# Verifica esplicita dell'aspect ratio 2:1
is_erp_2to1 = abs((width / height) - 2.0) < 0.01
print(f"Rispetta la convenzione ERP 2:1? {is_erp_2to1}")

cap.release()

### Nota

L'aspect ratio 2:1 non è una convenzione arbitraria: discende direttamente dalla geometria della proiezione equirettangolare.

#### La ragione geometrica

L'ERP mappa linearmente:

* la longitudine, che copre l'intero giro attorno alla sfera: 360°
* la latitudine, che copre solo la metà di un giro (dal polo sud al polo nord): 180°

Poiché un grado di longitudine occupa lo stesso numero di pixel di un grado di latitudine, il rapporto tra i due intervalli angolari, $360°/180° = 2$, diventa il rapporto tra larghezza e altezza del frame: $W = 2H$.

## 2. Estrazione di frame rappresentativi

Estraiamo alcuni frame a intervalli regolari, secondo lo stesso approccio di campionamento temporale già usato in Lezione 4, e li visualizziamo per osservare **empiricamente** — non genericamente — le distorsioni introdotte dalla proiezione ERP sul contenuto specifico di questo video.

In [ ]:
# Estrazione di frame a intervalli regolari (campionamento a passo fisso)
N_SAMPLES = 4
cap = cv2.VideoCapture(ERP_VIDEO_PATH)
n_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

sample_indices = np.linspace(0, n_total_frames - 1, N_SAMPLES, dtype=int)
sampled_frames = []

for idx in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))  # posiziona il cursore sul frame idx
    ret, frame = cap.read()
    if not ret:
        print(f"Attenzione: lettura fallita al frame {idx}")
        continue
    sampled_frames.append((int(idx), cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

cap.release()
print(f"Estratti {len(sampled_frames)} frame agli indici: {[i for i, _ in sampled_frames]}")

fig, axes = plt.subplots(1, len(sampled_frames), figsize=(16, 4))
for ax, (idx, frame) in zip(axes, sampled_frames):
    ax.imshow(frame)
    ax.set_title(f"Frame {idx}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Osservazione empirica sulla distorsione polare

Nella proiezione ERP la calotta superiore e quella inferiore della sfera occupano **l'intera larghezza del frame**: ogni riga di pixel copre sempre 360° di longitudine, a qualunque latitudine. Una fascia di 90° di longitudine occupa quindi sempre un quarto della larghezza del frame, sia all'equatore sia vicino ai poli. Solo all'equatore, però, questa larghezza corrisponde alla reale estensione della fascia sulla sfera: verso i poli la stessa fascia rappresenta una superficie sempre più piccola, che nell'ERP viene **stirata orizzontalmente**.

Nei frame estratti sopra, osserva le righe più in alto e più in basso: individua gli elementi della scena vicini allo zenit e al nadir (ad esempio cielo, soffitto o terreno) e confronta il loro aspetto con quello degli oggetti vicini all'orizzonte.

L'immagine seguente mostra lo stesso effetto su un altro contenuto ERP.

![EPR IMG](epr_img_small.jpg)

### Quantificare la distorsione

Possiamo rendere esplicito il meccanismo con un calcolo. Nell'ERP la larghezza in pixel di una fascia di longitudine è **costante** a ogni latitudine, perché la proiezione è lineare. L'area sferica reale rappresentata dalla stessa fascia, invece, è proporzionale a $\cos(\text{latitudine})$ e tende a zero ai poli. Il rapporto $1/\cos(\text{latitudine})$ indica quindi di quanto l'ERP **sovracampiona** in orizzontale quella latitudine rispetto all'equatore.

Prendiamo una fascia di 30° di longitudine a diverse latitudini e confrontiamo le due grandezze su un frame ERP di 1024×512 pixel.

In [ ]:
W, H = 1024, 512  # 2:1 -> ERP standard


def lonlat_to_xy(lon_deg, lat_deg, w=W, h=H):
    """Converte coordinate sferiche (longitudine, latitudine in gradi)
    nelle coordinate pixel (x, y) corrispondenti in un frame ERP di
    dimensioni w x h. Longitudine in [-180, 180], latitudine in [-90, 90]."""
    x = int((lon_deg + 180) / 360 * w) % w 
    y = int((90 - lat_deg) / 180 * h)
    y = min(max(y, 0), h - 1)  # clip di sicurezza sui bordi
    return x, y


print(f"{'Latitudine':>10} | {'px per 30° di lon.':>18} | {'area relativa cos(lat)':>22} | {'sovracampionamento 1/cos(lat)':>29}")
for lat_test in [0, 30, 60, 80]:
    x0, _ = lonlat_to_xy(-15, lat_test)
    x1, _ = lonlat_to_xy(15, lat_test)
    width_px = (x1 - x0) if x1 > x0 else (x1 - x0 + W)
    rel_area = math.cos(math.radians(lat_test))
    print(f"{lat_test:>9}° | {width_px:>18} | {rel_area:>22.3f} | {1 / rel_area:>28.1f}x")

La larghezza in pixel resta costante, mentre l'area sferica rappresentata decresce con $\cos(\text{latitudine})$: a 80° di latitudine la stessa fascia di pixel rappresenta meno di un quinto dell'area che rappresenterebbe all'equatore. In altre parole, a quella latitudine l'ERP spende quasi sei volte i pixel necessari.

## 3. Dalla ERP al cubemap con `py360convert`

### Richiamo teorico: perché il cubemap

Come osservato sopra, l'ERP campiona la sfera in modo **fortemente non uniforme**: la densità di pixel per unità di superficie sferica cresce man mano che ci si avvicina ai poli. Questo non è solo un problema estetico: significa che stiamo "sprecando" bit per codificare informazione ridondante ai poli, mentre l'equatore (spesso la zona di maggiore interesse visivo) non riceve un trattamento privilegiato.

La proiezione **cubemap** affronta il problema diversamente: la sfera viene proiettata sulle **6 facce di un cubo** circoscritto alla sfera (front, right, back, left, up, down). Ogni faccia è una proiezione **prospettica** (non equirettangolare) di 90°×90° di campo visivo, con una distorsione molto più uniforme e limitata alla propria faccia, invece che concentrata ai poli come nell'ERP.

![](hdri2cubemap.png)

### Nota

Il cubemap non elimina la distorsione — nessuna proiezione di una sfera su un piano può farlo (è lo stesso problema della cartografia terrestre) — ma la **ridistribuisce**: cresce verso gli angoli di ciascuna faccia, invece che concentrarsi tutta ai due poli come nell'ERP. Questa proprietà è alla base di molte tecniche di **codifica e streaming adattivo** su cui torneremo nella domanda di riflessione finale.

### Conversione con `e2c`

La funzione `py360convert.e2c` riceve un'immagine ERP in RGB e restituisce le facce del cubo. I parametri che usiamo sono:

- `face_w`: lato (in pixel) di ciascuna faccia quadrata;
- `mode`: metodo di interpolazione usato per il campionamento (qui bilineare);
- `cube_format="dict"`: restituisce un dizionario `{nome: faccia}`, comodo per accedere alle singole facce per nome.

Per scegliere `face_w` ricordiamo che una faccia copre 90° di longitudine: all'equatore la sorgente dispone quindi di circa `larghezza/4` pixel per faccia. Valori più piccoli sottocampionano il frame, valori più grandi non aggiungono informazione. Misuriamo il tempo di conversione sia con una faccia ridotta (256 px, sufficiente per visualizzare) sia alla risoluzione nativa.

In [ ]:
# Carichiamo il frame di test e lo convertiamo in RGB
# (OpenCV usa BGR internamente, py360convert e matplotlib si aspettano RGB)
erp_frame_bgr = cv2.imread(ERP_IMAGE_PATH)
erp_frame_rgb = cv2.cvtColor(erp_frame_bgr, cv2.COLOR_BGR2RGB)
print("Shape del frame ERP:", erp_frame_rgb.shape)

FACE_W = 256
FACE_W_NATIVE = erp_frame_rgb.shape[1] // 4


def timed_e2c(img, face_w):
    t0 = time.perf_counter()
    faces = py360convert.e2c(img, face_w=face_w, mode="bilinear", cube_format="dict")
    return faces, (time.perf_counter() - t0) * 1000


cube_faces, ms_small = timed_e2c(erp_frame_rgb, FACE_W)
_, ms_native = timed_e2c(erp_frame_rgb, FACE_W_NATIVE)

print(f"Conversione con face_w={FACE_W}: {ms_small:.1f} ms")
print(f"Conversione con face_w={FACE_W_NATIVE} (risoluzione nativa): {ms_native:.1f} ms")
print(f"Stima per l'intero video ({n_total_frames} frame) alla risoluzione nativa: "
      f"{ms_native * n_total_frames / 1000:.0f} s")
print("Facce prodotte:", list(cube_faces.keys()))
for name, face in cube_faces.items():
    print(f"  faccia '{name}': shape={face.shape}, dtype={face.dtype}")

In [ ]:
# Sviluppo "a croce" del cubemap: posizione (riga, colonna) di ogni faccia
FACE_ORDER = {
    "U": (0, 1), "L": (1, 0), "F": (1, 1),
    "R": (1, 2), "B": (1, 3), "D": (2, 1),
}


def show_cube_cross(faces, title):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax in axes.flat:
        ax.axis("off")
    for name, (r, c) in FACE_ORDER.items():
        ax = axes[r, c]
        ax.imshow(faces[name])
        ax.set_title(name)
        ax.axis("on")
        ax.set_xticks([])
        ax.set_yticks([])
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_cube_cross(cube_faces, "Frame reale: sviluppo a croce del cubemap (F=Front, R=Right, B=Back, L=Left, U=Up, D=Down)")

### Osservazione empirica

Osserva le facce **U** e **D**. Le regioni vicine allo zenit e al nadir, che nei frame della Sezione 2 erano stirate su tutta la larghezza del frame, qui sono raccolte in una sola faccia: confronta la loro forma con quella che avevano nell'ERP. È la ridistribuzione della distorsione descritta sopra: le calotte polari, che nell'ERP occupano due fasce a piena larghezza, nel cubemap occupano la porzione centrale di una faccia.

Sulle facce laterali (F, R, B, L) la scena appare come in una fotografia scattata con un obiettivo grandangolare. Cerca un elemento rettilineo della scena che nell'ERP appariva curvo e verifica come appare qui; osserva poi come cambiano le proporzioni degli oggetti avvicinandosi agli angoli di ciascuna faccia. Ne parliamo nella Sezione 4.

Infine, confronta i due tempi di conversione misurati sopra. Con `face_w=256` la conversione è rapida, ma stiamo sottocampionando il frame. Il tempo alla risoluzione nativa, moltiplicato per il numero di frame, dà l'ordine di grandezza del costo per l'**intero video**: è il dato da confrontare con i vincoli della pipeline (elaborazione offline o in tempo reale). Il valore dipende dall'hardware su cui gira il notebook.

## 4. Estrazione di viewport arbitrarie

### Richiamo teorico: cos'è una viewport

Quando un utente guarda un video 360° con un visore VR (o trascina il video con il mouse in un player desktop), **non vede mai il frame ERP o il cubemap direttamente**: il player genera al volo una **proiezione prospettica**, la viewport, corrispondente alla direzione in cui l'utente sta guardando. La viewport è definita dall'orientamento della testa (tre angoli) e dall'ampiezza del campo visivo:

- **yaw**: rotazione orizzontale (sinistra/destra), corrisponde alla longitudine;
- **pitch**: rotazione verticale (su/giù), corrisponde alla latitudine;
- **roll**: rotazione attorno all'asse di vista, inclina l'orizzonte senza cambiare il punto osservato;
- **FOV (field of view)**: ampiezza angolare del campo visivo simulato (tipicamente 90°–110° per un visore VR).

La viewport è quindi concettualmente **una faccia "orientabile" del cubemap**: con FOV di 90° e yaw/pitch multipli di 90° coincide con una delle 6 facce, ma si può calcolare per qualunque orientamento.

![Rotations](rotations.jpg)

In `py360convert.e2p` i quattro parametri corrispondono a:

- `u_deg`: yaw (0 = centro del frame ERP, valori positivi verso destra);
- `v_deg`: pitch (0 = equatore, valori positivi verso l'alto, 90 = polo nord);
- `in_rot_deg`: roll;
- `fov_deg`: FOV, come singolo valore o come tupla (orizzontale, verticale);
- `out_hw`: dimensioni (altezza, larghezza) dell'immagine di output.

In [ ]:
VIEWPORTS = {
    "Frontale\n(yaw=0, pitch=0)": (0, 0, 0),
    "Laterale destra\n(yaw=90, pitch=0)": (90, 0, 0),
    "Verso l'alto\n(yaw=0, pitch=60)": (0, 60, 0),
    "Posteriore\n(yaw=180, pitch=0)": (180, 0, 0),
    "Frontale con roll\n(yaw=0, pitch=0, roll=30)": (0, 0, 30),
}


def show_viewports(erp_img, title, fov=90, out_hw=(300, 300)):
    fig, axes = plt.subplots(1, len(VIEWPORTS), figsize=(20, 4.8))
    for ax, (label, (yaw, pitch, roll)) in zip(axes, VIEWPORTS.items()):
        vp = py360convert.e2p(
            erp_img,
            fov_deg=fov,
            u_deg=yaw,
            v_deg=pitch,
            in_rot_deg=roll,
            out_hw=out_hw,
        )
        ax.imshow(vp)
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_viewports(erp_frame_rgb, "Viewport estratte dal frame reale", out_hw=(600, 600))

### Esercizio guidato

Osserva le viewport del frame reale e confrontale con il frame ERP.

- La proiezione prospettica (gnomonica) usata per la viewport trasforma in rette esattamente i **cerchi massimi** della sfera, come i meridiani e l'equatore. Per questo gli spigoli rettilinei della scena tornano dritti nella viewport, mentre nell'ERP gli spigoli orizzontali lontani dall'orizzonte appaiono curvi: è l'ERP, non la viewport, a incurvare le rette. Cerca nella scena un elemento rettilineo (il bordo di un edificio, di una strada, di un tetto) e confronta il suo aspetto nell'ERP e nella viewport.
- Verso i bordi della viewport gli oggetti appaiono allargati. È la distorsione prospettica tipica di un obiettivo grandangolare rettilineo, che **stira** l'immagine ma non incurva le rette: un fenomeno diverso dalla distorsione di campionamento dell'ERP.
- Nella viewport "verso l'alto" la zona vicina allo zenit, che nell'ERP occupava una fascia a piena larghezza, appare come una regione **compatta**: la stessa geometria vista sulla faccia U del cubemap.
- Con il roll l'orizzonte si inclina, ma il punto osservato al centro della viewport non cambia.


### Esercizio in aula

Simula un utente che ruota lentamente la testa verso destra: estrai dal frame reale 6 viewport con yaw da 0° a 150° a passi di 30° (pitch=0°, FOV=90°) e mostrale affiancate. Poi ripeti con pitch=45°: cosa cambia nell'aspetto della scena?

È la versione ridotta dell'esercizio per casa.

In [ ]:
# Esercizio in aula: completa il codice
yaws = list(range(0, 180, 30))

fig, axes = plt.subplots(1, len(yaws), figsize=(18, 3.5))
for ax, yaw in zip(axes, yaws):
    vp = np.zeros((300, 300, 3), np.uint8)  # TODO: sostituisci con py360convert.e2p(...)
    ax.imshow(vp)
    ax.set_title(f"yaw={yaw}°")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Riepilogo ed esercizio per casa

### Riepilogo

- La proiezione **equirettangolare (ERP)** mappa linearmente longitudine/latitudine su x/y, con aspect ratio 2:1, ma campiona la sfera in modo non uniforme (sovracampionamento ai poli).
- La proiezione **cubemap** ridistribuisce questa distorsione su 6 facce prospettiche, in modo più uniforme.
- Una **viewport** è una proiezione prospettica arbitraria (yaw, pitch, roll, FOV), concettualmente una faccia "orientabile" del cubemap: è ciò che un utente vede effettivamente in un player 360°/VR.
- Nella viewport i **cerchi massimi** della sfera, e quindi gli spigoli rettilinei della scena, tornano rette: le curvature viste nell'ERP sono un effetto della proiezione equirettangolare, non della scena.

### Esercizio per casa (facoltativo)

Utilizzando `py360convert.e2p`, genera una sequenza di viewport estratte dallo stesso frame ERP con **yaw crescente** (ad esempio da 0° a 360° a passi di 10°, mantenendo pitch=0 e FOV=90°), simulando il movimento della testa di un utente che si guarda attorno. Componi le viewport risultanti in una breve clip video con `cv2.VideoWriter` (riutilizzando la correzione del codec vista in Lezione 4).

**Domanda di riflessione da preparare per la prossima lezione teorica:** se dovessi progettare uno schema di codifica *viewport-adaptive* per lo streaming live di un evento 360° (es. un concerto), quali informazioni ti servirebbero in tempo reale sul comportamento degli utenti, e quali compromessi dovresti accettare rispetto a una codifica uniforme dell'intero frame ERP?